<a href="https://colab.research.google.com/github/ozodbekAI/Data-Scince-and-AI-Portfolio/blob/main/RNN_IM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx

In [ ]:
!wget https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/test.xlsx

In [ ]:
import pandas as pd

df = pd.read_excel("train.xlsx")
df

In [ ]:
import torch
from collections import Counter

def tokenize(text):
  return str(text).lower().split()


counter = Counter(word for text in df['Reviews'] for word in tokenize(text))
most_common = counter.most_common(20000)
vocab = {word: i+2 for i, (word, _) in enumerate(most_common)}
vocab['<unk>'] = 0
vocab['<pad>'] = 1

def encode(text):
  return torch.tensor([vocab.get(word, 0) for word in tokenize(text)], dtype=torch.long)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
  def __init__(self, df):
    self.X = [encode(text) for text in df['Reviews']]
    self.y = [torch.tensor([1.0 if s == 'pos' else 0.0]) for s in df['Sentiment']]


  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [ ]:
def collate_fn(batch):
  Xs, ys = zip(*batch)
  max_len = max(len(x) for x in Xs)
  padded_Xs = [torch.cat([x, torch.zeros(max_len - len(x), dtype=torch.long)]) for x in Xs]
  return torch.stack(padded_Xs), torch.stack(ys)

train_loader = DataLoader(IMDBDataset(df), batch_size=16, shuffle=True, collate_fn=collate_fn)

In [ ]:
import torch.nn as nn

class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)
    _, h = self.rnn(x)
    x = self.fc(h.squeeze(0))
    return self.sigmoid(x)

In [ ]:
model = SentimentRNN(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(10):
  total_loss = 0

  for X, y in train_loader:
    y_pred = model(X)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

In [ ]:
def predict(text):
  model.eval()

  with torch.no_grad():
    encoded_text = encode(text)

    max_len_predict = 100

    if len(encoded_text) > max_len_predict:
      processed_text = encoded_text[:max_len_predict]

    elif len(encoded_text) < max_len_predict:
      padding = torch.full((max_len_predict - len(encoded_text),), vocab['<pad>'], dtype=torch.long)
      processed_text = torch.cat([encoded_text, padding])
    else:
      processed_text = encoded_text


    x = processed_text.unsqueeze(0)

    output = model(x)
    return output.item()

In [ ]:
print(f"{predict("I like this movie")}")